# Token Prediction — Disjointness and Independence

In a language model, consider the following events for a token:

- `A`: Token is capitalized.
- `B`: Token is a stop word.

Questions:

1. Can these events be **disjoint**?
2. Can these events be **independent**?
3. Compute overlap statistics from a tokenized piece of text.


## Step 1: Define the Events

Let:

- `A` = token is capitalized
- `B` = token is a stop word

Examples:

| Token | Capitalized? | Stop Word? |
|-------|--------------|-------------|
| `The` | Yes | Yes |
| `Python` | Yes | No |
| `the` | No | Yes |
| `database` | No | No |

Notice that `The` belongs to both classes.


## Step 2: Can the Events Be Disjoint?

Two events are **disjoint** if:

$$P(A \cap B)=0$$

For these events, the answer is:

**No.**

Words like:

- `The`
- `A`
- `An`
- `And`

can be both capitalized and stop words.

Therefore:

$$P(A \cap B) > 0$$

so the events are not disjoint.


## Step 3: Can the Events Be Independent?

Two events are independent if:

$$P(A \cap B)=P(A)P(B)$$

There is no theoretical reason why this must always hold.

Whether they are independent depends entirely on the corpus.

For example:

- A document containing many sentence starts may make stop words more likely to be capitalized.
- A document written in all lowercase may make capitalized tokens very rare.

Therefore, independence is an empirical question.


## Step 4: Sample Text

We'll use a short GPT-style completion and compute the statistics.


In [1]:
text = '''
The transformer architecture changed natural language processing.
It is now common to train large language models on vast amounts of text.
The model predicts the next token and then generates another token.
A language model can answer questions and write code.
'''

text

'\nThe transformer architecture changed natural language processing.\nIt is now common to train large language models on vast amounts of text.\nThe model predicts the next token and then generates another token.\nA language model can answer questions and write code.\n'

In [2]:
import re

tokens = re.findall(r"\b\w+\b", text)
tokens[:30]

['The',
 'transformer',
 'architecture',
 'changed',
 'natural',
 'language',
 'processing',
 'It',
 'is',
 'now',
 'common',
 'to',
 'train',
 'large',
 'language',
 'models',
 'on',
 'vast',
 'amounts',
 'of',
 'text',
 'The',
 'model',
 'predicts',
 'the',
 'next',
 'token',
 'and',
 'then',
 'generates']

In [3]:
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but",
    "is", "it", "to", "of", "on", "in",
    "for", "with", "at", "by", "from",
    "this", "that", "then", "can", "now"
}

In [4]:
import pandas as pd

df = pd.DataFrame(
    {
        "token": tokens
    }
)

df["capitalized"] = df["token"].str[0].str.isupper()
df["stop_word"] = (
    df["token"]
    .str.lower()
    .isin(STOP_WORDS)
)

df.head(20)

,token,capitalized,stop_word
0,The,True,True
1,transformer,False,False
2,architecture,False,False
3,changed,False,False
4,natural,False,False
5,language,False,False
6,processing,False,False
7,It,True,True
8,is,False,True
9,now,False,True


## Step 5: Compute Probabilities

In [5]:
p_capitalized = df["capitalized"].mean()

p_stop = df["stop_word"].mean()

p_both = (
    df["capitalized"]
    & df["stop_word"]
).mean()

pd.Series(
    {
        "P(Capitalized)": p_capitalized,
        "P(Stop Word)": p_stop,
        "P(Both)": p_both,
        "P(Capitalized) × P(Stop Word)": (
            p_capitalized * p_stop
        ),
    }
)

P(Capitalized)                   0.097561
P(Stop Word)                     0.341463
P(Both)                          0.097561
P(Capitalized) × P(Stop Word)    0.033314
dtype: float64

## Step 6: Check Disjointness

In [6]:
is_disjoint = p_both == 0
is_disjoint

np.False_

The answer should be `False` because some tokens belong to both classes.


## Step 7: Check Independence

In [7]:
tolerance = 1e-9

is_independent = (
    abs(
        p_both
        - p_capitalized * p_stop
    )
    <= tolerance
)

is_independent

np.False_

Usually this will evaluate to `False`.

The events are not theoretically constrained to be dependent or independent. Their relationship depends on the corpus.


## Step 8: Overlap Statistics

In [8]:
overlap = df[
    df["capitalized"]
    & df["stop_word"]
]

overlap["token"].value_counts()

token
The    2
It     1
A      1
Name: count, dtype: int64

## Final Answers

### Disjoint?

No.

Because:

$$P(A \cap B) > 0$$

Tokens such as `The` and `A` can be both capitalized and stop words.

### Independent?

Not necessarily.

Independence requires:

$$P(A \cap B)=P(A)P(B)$$

This must be verified empirically from the corpus.

The relationship between these events depends on the distribution of text being analyzed.
